#Teekide importimine

Impordime vajalikud teegid:
- pandas andmete töötlemiseks DataFrame kujul
- Supabase ühenduse loomiseks andmete laadimiseks
- dotenv keskkonnamuutujate lugemiseks

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from supabase import create_client

#Ühenduse loomine Supabase'iga

Loeme .env failist Supabase URL-i ja API võtme.
Neid kasutatakse turvalise ühenduse loomiseks andmebaasiga.

In [2]:
load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

#Andmete laadimine

Funktsioon get_data() laeb andmed Supabase tabelist pandas DataFrame'i.

Kasutame lehekülgede kaupa laadimist, sest Supabase tagastab vaikimisi piiratud koguse ridu.

In [3]:
response = supabase.table("sales").select("*").execute()

df_sales = pd.DataFrame(response.data)

df_sales.shape

(1000, 12)

In [4]:
def get_data(table_name):
    data = []
    page_size = 1000
    page = 0

    while True:
        response = supabase.table(table_name).select("*").range(
            page * page_size,
            (page + 1) * page_size - 1
        ).execute()

        data.extend(response.data)

        if len(response.data) < page_size:
            break

        page += 1

    return pd.DataFrame(data)

## Andmete laadimine

Laeme müügi- ja kliendiandmed eraldi DataFrame'idesse:
- df_sales
- df_customers

In [5]:
df_sales = get_data("sales")

df_sales.shape

(10118, 12)

In [6]:
df_sales.head()

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [7]:
df_sales.dtypes

id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object

In [8]:
df_customers = get_data("customers")

df_customers.shape

(0, 0)

#Mingil põhjusel ei näita customers tabelis andmeid, topelt kontollin.

In [9]:
response = supabase.table("customers").select("*").limit(5).execute()

response.data

[]

In [10]:
supabase.table("customers").select("*").execute()

APIResponse(data=[], count=None)

In [11]:
response = supabase.table("customers").select("customer_id").limit(5).execute()

response.data

[]

#Viga leitud. RLS supabasest eemaldatud.

In [12]:
response = supabase.table("customers").select("*").limit(5).execute()

response.data

[{'customer_id': 2637,
  'first_name': 'Reet',
  'last_name': 'Nurk',
  'email': None,
  'phone': '+372 5354 8280',
  'city': 'Tallinn',
  'registration_date': '2022-12-09',
  'loyalty_tier': 'gold',
  'birth_year': 1976},
 {'customer_id': 2723,
  'first_name': 'Reet',
  'last_name': 'Kuusik',
  'email': None,
  'phone': '+372 5603 8700',
  'city': 'Tartu',
  'registration_date': '2023-04-09',
  'loyalty_tier': None,
  'birth_year': 1998},
 {'customer_id': 2784,
  'first_name': 'Mart',
  'last_name': 'Pihl',
  'email': None,
  'phone': '+372 5536 5657',
  'city': 'Tallinn',
  'registration_date': '2022-10-30',
  'loyalty_tier': 'gold',
  'birth_year': 1989},
 {'customer_id': 3404,
  'first_name': 'Maie',
  'last_name': 'Tammik',
  'email': None,
  'phone': '+372 5291 9215',
  'city': 'Tallinn',
  'registration_date': '2020-03-26',
  'loyalty_tier': None,
  'birth_year': 2000},
 {'customer_id': 4278,
  'first_name': 'Nele',
  'last_name': 'Orav',
  'email': None,
  'phone': '+372 8700 9

In [13]:
df_customers = get_data("customers")

df_customers.shape

(3150, 9)

In [14]:
df_customers.head()

,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


#Tabelite ühendamine

Ühendame müügi- ja kliendiandmed customer_id põhjal.
Tulemuseks on üks DataFrame, kus on koos ostuinfo ja kliendiinfo.

In [15]:
df = pd.merge(
    df_sales,
    df_customers,
    on="customer_id",
    how="left"
)

df.shape

(10118, 20)

In [16]:
df_customers.shape

(3150, 9)

#Andmete kontroll

Kontrollime ühendatud tabeli suurust ja struktuuri:
- shape näitab ridade ja veergude arvu
- dtypes näitab andmetüüpe
- head() näitab esimesi ridu

In [18]:
df.head()

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [19]:
df.dtypes

id                     int64
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object

In [20]:
df.shape

(10118, 20)

In [23]:
df[["customer_id", "sale_date", "total_price", "email"]].head()

,customer_id,sale_date,total_price,email
0,2588.0,2023-01-10T00:00:00,469.58,NaN
1,4338.0,2023-01-16T00:00:00,482.26,merle.luik@mail.ee
2,4673.0,2023-01-05T00:00:00,221.19,liina.saar@gmail.com
3,4677.0,2023-01-02T00:00:00,135.63,aili.pihl@yahoo.com
4,2390.0,2023-01-13T00:00:00,99.57,triin.lill@telia.ee
